In [ ]:
"""
Support Vector Machines, implemented from scratch.

Two implementations:
  - LinearSVM  : primal formulation, trained via subgradient descent
                 on hinge loss + L2 regularization. O(n*d) per epoch.
                 Fast, good for large/sparse/high-dimensional data.
  - KernelSVM  : dual formulation, trained via projected gradient
                 ascent on the box-constrained QP. Supports nonlinear
                 kernels (linear, polynomial, RBF). Teaching-scale only
                 (O(n^2) memory for the kernel matrix, not SMO) --
                 fine for small datasets, not production use.

Also includes: kernel functions, support-vector identification,
and a couple of smoke tests you can run directly.
"""

import math
import random



# small helpers


def dot(a, b):
    return sum(ai * bi for ai, bi in zip(a, b))


def accuracy(y_true, y_pred):
    correct = sum(1 for a, b in zip(y_true, y_pred) if a == b)
    return correct / len(y_true)


def standardize(X):
    """Zero-mean, unit-variance scaling. Returns (X_scaled, means, stds).
    SVMs are sensitive to feature scale -- always standardize first."""
    n, d = len(X), len(X[0])
    means = [sum(row[j] for row in X) / n for j in range(d)]
    stds = []
    for j in range(d):
        var = sum((row[j] - means[j]) ** 2 for row in X) / n
        stds.append(math.sqrt(var) if var > 1e-12 else 1.0)
    X_scaled = [[(row[j] - means[j]) / stds[j] for j in range(d)] for row in X]
    return X_scaled, means, stds



# hinge loss


def hinge_loss(X, y, w, b):
    """Average hinge loss: max(0, 1 - y*(w.x + b)), zero once a point
    is correctly classified and outside the margin."""
    n = len(X)
    total_loss = sum(
        max(0.0, 1.0 - y[i] * (dot(w, X[i]) + b))
        for i in range(n)
    )
    return total_loss / n



# Linear SVM (primal, gradient descent)


class LinearSVM:
    """
    Minimizes:  (lambda/2)*||w||^2 + (1/n) * sum(hinge_loss)

    via per-sample subgradient descent (Pegasos-style updates).
    """

    def __init__(self, lr=0.001, lambda_param=0.01, n_epochs=1000):
        self.lr = lr
        self.lambda_param = lambda_param
        self.n_epochs = n_epochs
        self.w = None
        self.b = 0.0

    def fit(self, X, y):
        n_features = len(X[0])
        self.w = [0.0] * n_features
        self.b = 0.0

        for _epoch in range(self.n_epochs):
            for i in range(len(X)):
                margin = y[i] * (dot(self.w, X[i]) + self.b)
                if margin >= 1:
                    # only the regularization term contributes
                    self.w = [wj - self.lr * self.lambda_param * wj
                              for wj in self.w]
                else:
                    # regularization + hinge loss gradient
                    self.w = [
                        wj - self.lr * (self.lambda_param * wj - y[i] * X[i][j])
                        for j, wj in enumerate(self.w)
                    ]
                    self.b -= self.lr * (-y[i])
        return self

    def decision_function(self, X):
        return [dot(self.w, x) + self.b for x in X]

    def predict(self, X):
        return [1 if score >= 0 else -1 for score in self.decision_function(X)]


def find_support_vectors(X, y, w, b, tol=0.15):
    """Indices of points sitting (approximately) on the margin.

    Note: with a finite number of subgradient-descent epochs, w won't
    converge exactly to the analytic optimum, so an exact tol=1e-3 test
    against margin==1 can miss real support vectors. A looser tolerance
    (default 0.15) is more realistic for a non-QP-solver training loop.
    Points with margin close to 1 -- not comfortably above it -- are the
    ones "supporting" the boundary.
    """
    support_vectors = []
    for i in range(len(X)):
        margin = y[i] * (dot(w, X[i]) + b)
        if abs(margin - 1.0) < tol:
            support_vectors.append(i)
    return support_vectors



# Kernels


def linear_kernel(x, z):
    return dot(x, z)


def polynomial_kernel(x, z, degree=3, c=1.0):
    return (dot(x, z) + c) ** degree


def rbf_kernel(x, z, gamma=0.5):
    diff = [xi - zi for xi, zi in zip(x, z)]
    return math.exp(-gamma * dot(diff, diff))



# Kernel SVM (dual, projected gradient ascent)


class KernelSVM:
    """
    Dual-formulation SVM solved with projected gradient ascent on:

        maximize    sum(alpha_i) - 0.5 * sum_ij(alpha_i alpha_j y_i y_j K(x_i, x_j))
        subject to  0 <= alpha_i <= C
                    sum(alpha_i y_i) = 0

    NOTE: this is a simple, from-scratch solver meant for learning --
    NOT a replacement for a real implementation (libsvm uses SMO, which
    converges faster and more reliably). Limitations to be aware of:
      - builds the full n x n kernel matrix up front  -> O(n^2) memory,
        impractical past a few hundred/thousand points
      - fixed number of iterations, no real convergence check
      - the equality-constraint projection is a simple correction step,
        not an exact QP projection, so alpha can drift slightly
    Fine for small/teaching-scale datasets; don't use it in production.
    """

    def __init__(self, kernel=rbf_kernel, C=1.0, n_iters=500, lr=0.001, tol=1e-4):
        self.kernel = kernel
        self.C = C
        self.n_iters = n_iters
        self.lr = lr
        self.tol = tol
        self.alpha = None
        self.b = 0.0
        self.X = None
        self.y = None

    def fit(self, X, y):
        n = len(X)
        self.X = X
        self.y = y
        self.alpha = [0.0] * n

        # Precompute kernel matrix once -- O(n^2) memory, see class docstring
        K = [[self.kernel(X[i], X[j]) for j in range(n)] for i in range(n)]

        for _iteration in range(self.n_iters):
            # gradient ascent step, projected into the box [0, C]
            for i in range(n):
                grad = 1.0 - y[i] * sum(
                    self.alpha[j] * y[j] * K[i][j] for j in range(n)
                )
                self.alpha[i] += self.lr * grad
                self.alpha[i] = min(max(self.alpha[i], 0.0), self.C)

            # project onto the equality constraint sum(alpha_i * y_i) = 0
            correction = sum(self.alpha[i] * y[i] for i in range(n)) / n
            self.alpha = [
                min(max(self.alpha[i] - correction * y[i], 0.0), self.C)
                for i in range(n)
            ]

        # bias: average over support vectors strictly inside the box (0 < alpha < C)
        sv_indices = [i for i in range(n) if self.tol < self.alpha[i] < self.C - self.tol]
        if not sv_indices:
            sv_indices = [i for i in range(n) if self.alpha[i] > self.tol]

        b_values = []
        for i in sv_indices:
            s = sum(self.alpha[j] * y[j] * K[i][j] for j in range(n))
            b_values.append(y[i] - s)
        self.b = sum(b_values) / len(b_values) if b_values else 0.0

        return self

    def decision_function(self, X):
        scores = []
        for x in X:
            s = sum(
                self.alpha[i] * self.y[i] * self.kernel(self.X[i], x)
                for i in range(len(self.X))
                if self.alpha[i] > self.tol
            )
            scores.append(s + self.b)
        return scores

    def predict(self, X):
        return [1 if score >= 0 else -1 for score in self.decision_function(X)]

    def support_vector_indices(self):
        return [i for i in range(len(self.alpha)) if self.alpha[i] > self.tol]



# Smoke tests


if __name__ == "__main__":
    random.seed(42)

    # --- Linear SVM: two well-separated blobs ---
    X_lin, y_lin = [], []
    for _ in range(30):
        X_lin.append([random.gauss(-2, 0.5), random.gauss(-2, 0.5)])
        y_lin.append(-1)
    for _ in range(30):
        X_lin.append([random.gauss(2, 0.5), random.gauss(2, 0.5)])
        y_lin.append(1)

    lin_svm = LinearSVM(lr=0.001, lambda_param=0.01, n_epochs=200).fit(X_lin, y_lin)
    preds = lin_svm.predict(X_lin)
    print("Linear SVM training accuracy:", accuracy(y_lin, preds))
    sv = find_support_vectors(X_lin, y_lin, lin_svm.w, lin_svm.b)
    print("Linear SVM support vector count:", len(sv))

    # --- Kernel SVM: concentric circles (not linearly separable) ---
    X_circ, y_circ = [], []
    for _ in range(40):
        angle = random.uniform(0, 2 * math.pi)
        r = random.uniform(0, 1.0)
        X_circ.append([r * math.cos(angle), r * math.sin(angle)])
        y_circ.append(-1)
    for _ in range(40):
        angle = random.uniform(0, 2 * math.pi)
        r = random.uniform(2.0, 3.0)
        X_circ.append([r * math.cos(angle), r * math.sin(angle)])
        y_circ.append(1)

    rbf_svm = KernelSVM(
        kernel=lambda x, z: rbf_kernel(x, z, gamma=0.5),
        C=1.0, n_iters=100, lr=0.01,
    ).fit(X_circ, y_circ)
    preds_circ = rbf_svm.predict(X_circ)
    print("RBF kernel SVM training accuracy (circles):", accuracy(y_circ, preds_circ))
    print("RBF kernel SVM support vector count:", len(rbf_svm.support_vector_indices()))


Linear SVM training accuracy: 1.0
Linear SVM support vector count: 3
RBF kernel SVM training accuracy (circles): 1.0
RBF kernel SVM support vector count: 79
